In [ ]:
# === Imports & EOS-80 density helper =========================================
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import cm, colors
from matplotlib.lines import Line2D
import tfv.xarray  # registers .tfv accessor

def eos80_potential_density(S, T):
    """UNESCO EOS-80 potential density (p=0). S psu, T degC. Vectorised."""
    T2, T3, T4, T5 = T*T, T*T*T, T*T*T*T, T*T*T*T*T
    Ssq = np.sqrt(np.clip(S, 0, None)); S1p5 = S*Ssq; S2 = S*S
    a = [999.842594, 6.793952e-2, -9.095290e-3, 1.001685e-4, -1.120083e-6, 6.536332e-9]
    rho_w = a[0] + a[1]*T + a[2]*T2 + a[3]*T3 + a[4]*T4 + a[5]*T5
    b = [8.24493e-1, -4.0899e-3, 7.6438e-5, -8.2467e-7, 5.3875e-9]
    c = [-5.72466e-3, 1.0227e-4, -1.6546e-6]; d0 = 4.8314e-4
    return (rho_w + (b[0]+b[1]*T+b[2]*T2+b[3]*T3+b[4]*T4)*S
            + (c[0]+c[1]*T+c[2]*T2)*S1p5 + d0*S2)

In [ ]:
# === Configuration ===========================================================
MODEL_NC  = Path(r'S:/Matt_Working/csiem/output_archive/1.7.0/1991_aug/csiem_B010_19910720_19910831.nc')
TRANSECT_SHP = Path(r'G:/CSIEM/1.8.0/csiem-marvl/custom_py/dadamo_transect/transect_A_100m.shp')
OUT_DIR   = Path(r'G:/CSIEM/1.8.0/csiem-marvl/custom_py/dadamo_transect/outputs_1991_TransectA')
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_MODE = True   # True -> render just the first survey panel (all 3 vars)

VARIABLES = ['RHOW', 'SAL', 'TEMP']
VARSPECS = {
    'RHOW': dict(clim=(1024, 1027),  cmap='jet',      label='Water Density (kg m$^{-3}$)'),
    'SAL' : dict(clim=(34.8, 35.6),  cmap='viridis',  label='Salinity (psu)'),
    'TEMP': dict(clim=(15.0, 18.5),  cmap='RdYlBu_r', label='Temperature ($^\circ$C)'),
}
DEPTH_YLIM = (-24, 0)

# --- Transect A station coords (lat, lon), from SMCWS map_stations.py ---
COORDS = {
 'OA10':(-32.0558,115.7308),'OA15':(-32.0593,115.7155),'OA20':(-32.0663,115.7153),
 'OA25':(-32.0929,115.7153),'OA65':(-32.1113,115.7053),'OA80':(-32.1313,115.7013),
 'CS20':(-32.1501,115.7055),'CS45':(-32.1679,115.7095),'CS55':(-32.1876,115.7140),
 'CS85':(-32.2113,115.7192),'CS105':(-32.2346,115.7245),'CS135':(-32.2481,115.7008),
 'CS140':(-32.2489,115.6985),'CS145':(-32.2554,115.6845),'CS150':(-32.2554,115.6667),
 'CS155':(-32.2464,115.7207),
}
# Ordered NORTH -> SOUTH so the curtain reads N (left) -> S (right)
TRANSECT_A_NS = ['OA10','OA15','OA20','OA25','OA65','OA80','CS20','CS45','CS55',
                 'CS85','CS105','CS155','CS135','CS140','CS145','CS150']
station_points = {s: (COORDS[s][1], COORDS[s][0]) for s in TRANSECT_A_NS}  # (lon, lat)
sites = TRANSECT_A_NS

# --- Survey panel time windows (from SMCWS plan_transects.py) ---
PANELS = [
 ('6.16a','pre', datetime(1991,8,13,15,36), datetime(1991,8,13,17,27)),
 ('6.16b','pre', datetime(1991,8,13,20,9),  datetime(1991,8,14,0,25)),
 ('6.16c','pre', datetime(1991,8,14,11,50), datetime(1991,8,14,17,32)),
 ('6.16d','pre', datetime(1991,8,14,21,16), datetime(1991,8,15,0,59)),
 ('6.16e','pre', datetime(1991,8,15,9,39),  datetime(1991,8,15,15,50)),
 ('6.16f','pre', datetime(1991,8,15,19,20), datetime(1991,8,15,20,48)),
 ('6.16g','pre', datetime(1991,8,16,0,29),  datetime(1991,8,16,1,35)),
 ('6.16h','pre', datetime(1991,8,16,11,39), datetime(1991,8,16,13,35)),
 ('6.16i','pre', datetime(1991,8,16,19,4),  datetime(1991,8,16,23,32)),
 ('6.16j','pre', datetime(1991,8,17,1,44),  datetime(1991,8,17,2,41)),
 ('6.16k','pre', datetime(1991,8,17,10,25), datetime(1991,8,17,11,50)),
 ('6.16l','pre', datetime(1991,8,17,11,50), datetime(1991,8,17,15,46)),
 ('6.17a','post',datetime(1991,8,20,7,48),  datetime(1991,8,20,10,9)),
 ('6.17b','post',datetime(1991,8,20,19,31), datetime(1991,8,20,22,47)),
 ('6.17c','post',datetime(1991,8,21,2,27),  datetime(1991,8,21,3,13)),
 ('6.17d','post',datetime(1991,8,21,7,51),  datetime(1991,8,21,10,32)),
 ('6.17e','post',datetime(1991,8,21,18,50), datetime(1991,8,21,21,47)),
 ('6.17f','post',datetime(1991,8,22,0,45),  datetime(1991,8,22,2,37)),
 ('6.17g','post',datetime(1991,8,22,13,4),  datetime(1991,8,22,17,10)),
 ('6.17h','post',datetime(1991,8,22,23,1),  datetime(1991,8,23,1,59)),
]

In [ ]:
# === Load Transect A polyline + per-station chainage =========================
g = gpd.read_file(TRANSECT_SHP)
if g.crs is not None and g.crs.to_epsg() != 4326:
    g = g.to_crs(4326)
polyline = np.array(g.geometry.iloc[0].coords)
# shapefile was written S->N; flip to N->S to match TRANSECT_A_NS / curtain orientation
if polyline[0, 1] < polyline[-1, 1]:
    polyline = polyline[::-1].copy()

# chainage (m) along the polyline from the north end (= curtain x-axis)
_lat0 = polyline[:, 1].mean()
_mlon = 111320.0 * np.cos(np.deg2rad(_lat0)); _mlat = 111320.0
_seg = np.hypot(np.diff(polyline[:,0])*_mlon, np.diff(polyline[:,1])*_mlat)
_chain = np.concatenate(([0.0], np.cumsum(_seg)))
def station_chainage(lon, lat):
    d2 = ((polyline[:,0]-lon)*_mlon)**2 + ((polyline[:,1]-lat)*_mlat)**2
    return float(_chain[int(np.argmin(d2))])
station_chainage_m = {s: station_chainage(*station_points[s]) for s in sites}
print(f'polyline {len(polyline)} pts, length {_chain[-1]/1000:.1f} km (N->S)')
print('CS55 chainage from north = {:.0f} m'.format(station_chainage_m['CS55']))

In [ ]:
# === Open model output + inject density; build per-panel model times =========
ds = xr.open_dataset(MODEL_NC)
ds['RHOW'] = eos80_potential_density(ds['SAL'], ds['TEMP'])
ds['RHOW'].attrs.update(long_name='Potential density (EOS-80, p=0)', units='kg/m3')
fv = ds.tfv
times = pd.to_datetime(ds['Time'].values)
tmin, tmax = times.min(), times.max()

panel_jobs = []  # (panel_id, phase, model_date_exact)
for pid, phase, t0, t1 in PANELS:
    mid = pd.Timestamp(t0) + (pd.Timestamp(t1) - pd.Timestamp(t0)) / 2
    if not (tmin <= mid <= tmax):
        print(f'  skip {pid}: midpoint {mid} outside model coverage')
        continue
    # snap to nearest model timestep: get_profile requires an EXACT model time
    md = times[int(np.argmin(np.abs(times - mid)))]
    panel_jobs.append((pid, phase, md))
print(f'Model coverage {tmin} -> {tmax}; {len(panel_jobs)}/{len(PANELS)} panels in range '
      f'(model times snapped to nearest 4-hourly step)')

In [ ]:
# === Figure builder (model-only): curtain + 16 station profiles + map ========
def make_panel_figure(var, panel_id, phase, model_date, save=True, show=False):
    spec = VARSPECS[var]; clim, cmap, clabel = spec['clim'], spec['cmap'], spec['label']
    y_min, y_max = DEPTH_YLIM

    fig = plt.figure(figsize=(17, 13))
    gs = fig.add_gridspec(5, 5, width_ratios=[1, 1, 1, 1, 0.85],
                          height_ratios=[1.3, 1, 1, 1, 1], hspace=0.30, wspace=0.12)
    ax_transect = fig.add_subplot(gs[0, :4])
    ax_cbar_slot = fig.add_subplot(gs[0, 4]); pos = ax_cbar_slot.get_position(); ax_cbar_slot.remove()
    ax_cbar = fig.add_axes([pos.x0 + pos.width*0.55, pos.y0 + pos.height*0.02,
                            pos.width*0.20, pos.height*0.95])
    ax_map = fig.add_subplot(gs[1:3, 4])
    # 16 profile panels in rows 1-4, cols 0-3
    profile_axes = []
    for r in range(1, 5):
        for c in range(4):
            axp = fig.add_subplot(gs[r, c], sharey=profile_axes[0] if profile_axes else None)
            profile_axes.append(axp)

    # ---- curtain ----
    try:
        cs = fv.plot_curtain(polyline, var, time=model_date, ax=ax_transect,
                             ec='face', clim=clim, colorbar=False, cmap=cmap)
        try:
            cbar = fig.colorbar(cs, cax=ax_cbar, orientation='vertical')
        except Exception:
            sm = cm.ScalarMappable(norm=colors.Normalize(*clim), cmap=cmap); sm.set_array([])
            cbar = fig.colorbar(sm, cax=ax_cbar, orientation='vertical')
        cbar.set_label(clabel, fontsize=9); cbar.ax.tick_params(labelsize=8)
        y_top = max(ax_transect.get_ylim())
        for name in sites:
            x_ch = station_chainage_m[name]
            ax_transect.axvline(x_ch, color='white', ls='--', lw=0.8, alpha=0.85, zorder=8)
            ax_transect.plot(x_ch, y_top, marker='v', ms=5, mfc='white', mec='black',
                             clip_on=False, zorder=9)
            ax_transect.text(x_ch, y_top + 0.3, name, fontsize=6.5, rotation=90,
                             va='bottom', ha='center', zorder=10)
        ax_transect.set_xlabel('Chainage from north (m)', fontsize=10)
        ax_transect.set_ylabel('Depth (m)', fontsize=10)
        ax_transect.set_title(f'Transect A | {var} | panel {panel_id} ({phase}-storm) | '
                              f'model {pd.Timestamp(model_date).strftime("%Y-%m-%d %H:%M")}',
                              fontsize=11, fontweight='bold', pad=18)
        ax_transect.text(0.01, 1.02, 'N', transform=ax_transect.transAxes, fontsize=12, fontweight='bold')
        ax_transect.text(0.99, 1.02, 'S', transform=ax_transect.transAxes, fontsize=12, fontweight='bold', ha='right')
    except Exception as e:
        ax_transect.text(0.5, 0.5, f'Curtain error: {e}', transform=ax_transect.transAxes, ha='center', va='center')
        ax_cbar.set_axis_off()

    # ---- model profiles ----
    for idx, site in enumerate(sites):
        ax = profile_axes[idx]
        try:
            prof = fv.get_profile(station_points[site], variables=[var], time=model_date)
            if prof is not None:
                pt = prof.sel(Time=model_date, method='nearest') if 'Time' in prof.dims else prof
                x_mod = np.asarray(pt[var]).ravel(); z_mod = np.asarray(pt['Z']).ravel()
                ok = np.isfinite(x_mod) & np.isfinite(z_mod)
                if ok.sum() > 0:
                    ax.plot(x_mod[ok], z_mod[ok], '-s', color='steelblue', alpha=0.8, lw=1.8, ms=2.5, label='Model')
        except Exception:
            pass
        ax.set_title(site, fontweight='bold', fontsize=9)
        ax.set_xlim(*clim); ax.set_ylim(y_min, y_max)
        ax.set_yticks(np.arange(-24, 2, 4)); ax.grid(True, alpha=0.3)
        ax.tick_params(axis='x', labelrotation=30, labelsize=7)
        if idx < 12:
            ax.tick_params(labelbottom=False)
        else:
            ax.set_xlabel(clabel, fontsize=8)
        if idx % 4 == 0:
            ax.set_ylabel('Depth (m)', fontsize=8)
        else:
            ax.tick_params(labelleft=False)

    # ---- map inset ----
    try:
        fv.plot(var, time=model_date, ax=ax_map, datum='depth', cmap=cmap, clim=clim,
                shading='interp', colorbar=False, boundary=True)
        ax_map.plot(polyline[:, 0], polyline[:, 1], 'r-', lw=1.8)
        for name, (lon, lat) in station_points.items():
            ax_map.plot(lon, lat, 'o', ms=3.5, mfc='yellow', mec='black', zorder=6)
        lon_min, lon_max = polyline[:,0].min(), polyline[:,0].max()
        lat_min, lat_max = polyline[:,1].min(), polyline[:,1].max()
        lon_pad = max((lon_max-lon_min)*0.12, 0.004); lat_pad = max((lat_max-lat_min)*0.05, 0.004)
        ax_map.set_xlim(lon_min-lon_pad, lon_max+lon_pad); ax_map.set_ylim(lat_min-lat_pad, lat_max+lat_pad)
        ax_map.set_aspect(1/np.cos(np.radians(32.15)))
        ax_map.grid(True, color='darkgrey', alpha=0.6, lw=0.5)
        ax_map.xaxis.set_major_locator(mticker.MaxNLocator(3)); ax_map.yaxis.set_major_locator(mticker.MaxNLocator(5))
        ax_map.tick_params(labelsize=7); ax_map.tick_params(axis='y', labelrotation=90)
        ax_map.set_title('Transect A', fontsize=9, fontweight='bold')
    except Exception as e:
        ax_map.text(0.5, 0.5, f'Map error: {e}', transform=ax_map.transAxes, ha='center', va='center'); ax_map.set_axis_off()

    fig.legend(handles=[Line2D([0],[0], color='steelblue', lw=2, marker='s', ms=5, label='Model')],
               loc='lower right', fontsize=9, frameon=True, bbox_to_anchor=(0.985, 0.02))
    fig.subplots_adjust(top=0.93, bottom=0.06, left=0.05, right=0.99)
    if save:
        fname = OUT_DIR / f'transectA_{var}_{panel_id}_{phase}.png'
        fig.savefig(fname, dpi=200, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return (OUT_DIR / f'transectA_{var}_{panel_id}_{phase}.png') if save else None

In [ ]:
# === Generate figures ========================================================
jobs = panel_jobs[:1] if TEST_MODE else panel_jobs
print(f'{"TEST" if TEST_MODE else "FULL"} run: {len(jobs)} panel(s) x {len(VARIABLES)} vars '
      f'= {len(jobs)*len(VARIABLES)} figures')
for var in VARIABLES:
    for pid, phase, mdate in jobs:
        out = make_panel_figure(var, pid, phase, mdate, save=True, show=TEST_MODE)
        print(f'  saved {out.name}')
print('Done.')